In [1]:
import numpy as np
import joblib
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from xgboost import XGBClassifier


In [2]:
MODEL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\models"
RESULTS_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\notebooks\results"

os.makedirs(RESULTS_DIR, exist_ok=True)

X = np.load(f"{MODEL_DIR}/cnnvit_features.npy")
y = np.load(f"{MODEL_DIR}/cnnvit_labels.npy")

print("Features:", X.shape)
print("Labels:", y.shape)


Features: (6939, 896)
Labels: (6939,)


In [3]:
# First split: Train+Val vs Test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Second split: Train vs Val
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.125,   # 0.125 * 0.8 = 0.10 total
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


Train: (4857, 896)
Val: (694, 896)
Test: (1388, 896)


In [5]:
np.save(f"{RESULTS_DIR}/X_train_cnnvit.npy", X_train)
np.save(f"{RESULTS_DIR}/X_val_cnnvit.npy", X_val)
np.save(f"{RESULTS_DIR}/X_test_cnnvit.npy", X_test)

np.save(f"{RESULTS_DIR}/y_train_cnnvit.npy", y_train)
np.save(f"{RESULTS_DIR}/y_val_cnnvit.npy", y_val)
np.save(f"{RESULTS_DIR}/y_test_cnnvit.npy", y_test)

print("Splits saved ")


Splits saved 


In [6]:
xgb_baseline = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

start_train = time.time()

xgb_baseline.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

train_time = time.time() - start_train

print("Training time:", round(train_time, 2), "seconds")


Training time: 85.67 seconds


In [7]:
start_test = time.time()

y_pred = xgb_baseline.predict(X_test)
y_prob = xgb_baseline.predict_proba(X_test)[:, 1]

test_time = time.time() - start_test

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

TPR = tp / (tp + fn)
FPR = fp / (fp + tn)

print("\n===== CNN + ViT + XGB (Before Optuna) =====")
print("Accuracy:", acc)
print("AUROC:", auc)
print("TPR:", TPR)
print("FPR:", FPR)
print("Testing time:", round(test_time, 4), "seconds")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



===== CNN + ViT + XGB (Before Optuna) =====
Accuracy: 0.9373198847262247
AUROC: 0.984046845102243
TPR: 0.9330143540669856
FPR: 0.05913272010512484
Testing time: 0.0778 seconds

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       761
           1       0.93      0.93      0.93       627

    accuracy                           0.94      1388
   macro avg       0.94      0.94      0.94      1388
weighted avg       0.94      0.94      0.94      1388



In [8]:
joblib.dump(xgb_baseline, f"{MODEL_DIR}/cnnvit_xgb_baseline.pkl")

np.save(f"{RESULTS_DIR}/cnnvit_baseline_y_pred.npy", y_pred)
np.save(f"{RESULTS_DIR}/cnnvit_baseline_y_prob.npy", y_prob)

print("Baseline model saved ")


Baseline model saved 


In [1]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

TPR = tp / (tp + fn)   # Sensitivity
FPR = fp / (fp + tn)   # False Positive Rate

print("TPR (Sensitivity):", TPR)
print("FPR:", FPR)


NameError: name 'y_test' is not defined